In [15]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import joblib
import warnings

warnings.filterwarnings("ignore")

In [16]:
X_train = joblib.load("Demand_X_train.pkl")
X_test = joblib.load("Demand_X_test.pkl")

y_train = joblib.load("Demand_y_train.pkl")
y_test = joblib.load("Demand_y_test.pkl")

In [17]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 8)
X_test  : (26, 8)
y_train : (100,)
y_test  : (26,)


In [18]:
# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest Model
rf = RandomForestRegressor(random_state=42)

# Hyperparameter Search Space
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [5, 8, 10, 15, 20, None],
    'min_samples_split': [2, 3, 5, 8],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features': ['sqrt', 'log2', 0.7, 1.0],
    'bootstrap': [True, False]
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

# Train Model
random_search.fit(X_train, y_train)

# Best Model
rf_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'n_estimators': 100, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 15, 'bootstrap': True}


In [19]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=15, min_samples_split=8, random_state=42)

In [20]:
y_pred = rf_model.predict(X_test)

In [21]:
print(y_pred)

[ 8136.68240349  8831.64947692  9001.25419144  9654.47629107
 10914.65905934 10900.98357449 10822.33498358 10577.07254502
 10637.24089819 10582.01193535 10734.83815804  9341.82038161
  8749.5838521   8682.73714993  8970.93619784  8987.82453211
 10839.7165206  10880.70845977 10846.81565422 10795.42921396
 10832.55263171 10651.14727737 10588.54649129  9456.51108072
  8918.20834205  8970.63405498]


In [22]:
prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
prediction_df.head(10)

,Actual,Predicted
0,8759.0,8136.682403
1,9356.0,8831.649477
2,9658.0,9001.254191
3,10404.0,9654.476291
4,10452.0,10914.659059
5,12649.0,10900.983574
6,11838.0,10822.334984
7,10784.0,10577.072545
8,11192.0,10637.240898
9,11135.0,10582.011935


In [23]:
# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Training Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Testing Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*50)
print("Training Performance")
print("="*50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape*100:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("="*50)
print("Testing Performance")
print("="*50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape*100:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 192.76
RMSE : 259.94
MAPE : 2.15%
R²   : 0.9231


Testing Performance
MAE  : 801.15
RMSE : 875.62
MAPE : 7.45%
R²   : 0.2069


In [24]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

            Feature  Importance
3       Temperature    0.485945
4              Year    0.294707
0          Humidity    0.085255
2  Solar_Irradiance    0.066636
6         Month_cos    0.036973
1          Rainfall    0.021418
5         Month_sin    0.008872
7          Festival    0.000195


In [25]:
print(X_train.describe())

         Humidity    Rainfall  Solar_Irradiance  Temperature        Year  \
count  100.000000  100.000000        100.000000   100.000000   100.00000   
mean    74.160400   95.606726        152.503900    26.950600  2019.16000   
std      7.578832   75.634605         23.807573     1.801522     2.44007   
min     56.410000    0.786286         79.840000    23.660000  2015.00000   
25%     69.722500   37.067500        139.827500    25.547500  2017.00000   
50%     74.520000   82.542714        155.470000    27.010000  2019.00000   
75%     79.937500  139.723929        167.277500    28.080000  2021.00000   
max     89.420000  399.657143        204.470000    31.110000  2023.00000   

          Month_sin     Month_cos    Festival  
count  1.000000e+02  1.000000e+02  100.000000  
mean  -3.232051e-02 -8.660254e-03    0.160000  
std    7.152420e-01  7.052652e-01    0.368453  
min   -1.000000e+00 -1.000000e+00    0.000000  
25%   -8.660254e-01 -5.915064e-01    0.000000  
50%   -2.449294e-16 -1.8369

In [26]:
print(X_test.describe())

        Humidity    Rainfall  Solar_Irradiance  Temperature         Year  \
count  26.000000   26.000000         26.000000    26.000000    26.000000   
mean   76.338077  107.520901        184.847123    26.957308  2024.384615   
std     7.761415   76.711440         60.849792     1.853549     0.637302   
min    56.850000    4.588857         98.660000    23.560000  2023.000000   
25%    72.005000   41.301643        128.440000    25.490000  2024.000000   
50%    76.485000   91.552000        186.560000    27.220000  2024.000000   
75%    82.562500  163.227786        228.634221    28.095000  2025.000000   
max    86.440000  257.944286        303.305385    31.320000  2025.000000   

          Month_sin     Month_cos   Festival  
count  2.600000e+01  2.600000e+01  26.000000  
mean  -1.923077e-02  7.177021e-02   0.192308  
std    6.997252e-01  7.379993e-01   0.401918  
min   -1.000000e+00 -1.000000e+00   0.000000  
25%   -5.000000e-01 -5.000000e-01   0.000000  
50%   -2.449294e-16  6.123234e-17

In [27]:
print(X_train.columns.tolist())

['Humidity', 'Rainfall', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Festival']


In [28]:
import joblib

# Save the trained Random Forest model
joblib.dump(rf_model, "RandomForest_Demand_Forecasting_08610.pkl")

print("Random Forest model saved successfully!")

Random Forest model saved successfully!
